# 🇹🇼 Kaggle Taiwan Stock Multi-Factor ML Ranker (LightGBM & XGBoost)
### Machine Learning Predictive Alpha & Forward 5-Day Return Ranking

This notebook implements:
1. **Taiwan High-Cap Universe Ingestion via yfinance** (TSMC 2330, MediaTek 2454, Foxconn 2317, Delta 2308, Fubon 2881, etc.)
2. **Cross-Sectional & Time-Series Quantitative Factor Engineering** (Momentum, Volatility, Moving Average Disparities, RSI, Volume Surges)
3. **Walk-Forward Cross-Validation** to eliminate lookahead bias
4. **Information Coefficient (Rank IC)** and Sharpe Ratio validation
5. **Feature Importance Analysis & Automated Portfolio Allocation**
6. **Exportable Model Artifacts (`taiwan_stock_lgbm.joblib`)**

In [ ]:
# Step 1: Install packages
!pip install yfinance lightgbm xgboost scikit-learn pandas numpy matplotlib joblib


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import lightgbm as lgb
import xgboost as xgb

UNIVERSE = ['2330.TW', '2317.TW', '2454.TW', '2308.TW', '2382.TW', '2881.TW', '2882.TW', '2603.TW']
print(f'Target Universe: {UNIVERSE}')


In [ ]:
# Step 2: Ingest Stock History Data
stock_dfs = {}
for sym in UNIVERSE:
    try:
        t = yf.Ticker(sym)
        h = t.history(period='2y')
        if len(h) >= 120:
            stock_dfs[sym] = h
            print(f'Fetched {sym}: {len(h)} rows')
    except Exception as e:
        print(f'Error fetching {sym}: {e}')


In [ ]:
# Step 3: Extract Multi-Factor Features
def extract_features(df, forward_period=5):
    df = df.copy()
    c = df['Close']
    h = df['High']
    l = df['Low']
    v = df['Volume']

    # Momentum
    df['mom_5d'] = c.pct_change(5)
    df['mom_10d'] = c.pct_change(10)
    df['mom_20d'] = c.pct_change(20)
    df['mom_60d'] = c.pct_change(60)

    # Disparity
    ma5 = c.rolling(5).mean()
    ma20 = c.rolling(20).mean()
    ma60 = c.rolling(60).mean()
    df['bias_ma5'] = (c - ma5) / (ma5 + 1e-9)
    df['bias_ma20'] = (c - ma20) / (ma20 + 1e-9)
    df['bias_ma60'] = (c - ma60) / (ma60 + 1e-9)

    # Volatility & Volume
    df['hist_vol_20'] = c.pct_change().rolling(20).std() * np.sqrt(250)
    df['vol_surge'] = v / (v.rolling(5).mean() + 1e-9)

    # RSI
    delta = c.diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    df['rsi_14'] = 100 - (100 / (1 + rs))

    # Target: 5-Day forward return
    df['target_fwd_5d'] = c.shift(-forward_period) / c - 1.0
    return df

panels = []
for sym, df in stock_dfs.items():
    feat = extract_features(df)
    feat['symbol'] = sym
    panels.append(feat)

combined = pd.concat(panels, axis=0)
exclude = {'Open', 'High', 'Low', 'Close', 'Volume', 'target_fwd_5d', 'symbol', 'Dividends', 'Stock Splits'}
features = [c for c in combined.columns if c not in exclude]
clean = combined.dropna(subset=features + ['target_fwd_5d']).copy()
print(f'Dataset ready: {len(clean)} rows, {len(features)} features: {features}')


In [ ]:
# Step 4: Walk-Forward Validation & Model Training
X = clean[features].values
y = clean['target_fwd_5d'].values

split = int(len(X) * 0.8)
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)
model.fit(X_train, y_train)

test_preds = model.predict(X_test)
rank_ic = pd.Series(test_preds).corr(pd.Series(y_test), method='spearman')
print(f'Out-of-sample Rank IC: {rank_ic:.4f}')

# Feature Importance
imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(10, 4))
imp.plot(kind='bar', color='teal')
plt.title('Taiwan Stock LightGBM Feature Importance')
plt.ylabel('Split Count')
plt.grid(True)
plt.show()


In [ ]:
# Step 5: Generate Live Predictions & Stock Rankings
rankings = []
for sym, df in stock_dfs.items():
    feat = extract_features(df)
    latest = feat.iloc[-1:]
    pred = float(model.predict(latest[features].values)[0])
    rankings.append({
        'Symbol': sym,
        'Latest Close': round(float(latest['Close'].iloc[0]), 2),
        'Predicted 5D Ret (%)': round(pred * 100, 2),
        'Bias MA20 (%)': round(float(latest['bias_ma20'].iloc[0]) * 100, 2),
        'RSI(14)': round(float(latest['rsi_14'].iloc[0]), 1)
    })

rank_df = pd.DataFrame(rankings).sort_values(by='Predicted 5D Ret (%)', ascending=False).reset_index(drop=True)
rank_df.index += 1
print('=== Top Predicted Taiwan Stocks for Next 5 Days ===')
print(rank_df)

# Step 6: Export Trained Weights
joblib.dump({'model': model, 'features': features}, 'taiwan_stock_lgbm.joblib')
print('Model successfully saved to taiwan_stock_lgbm.joblib')
